# Synthetic Data Generator
### Step 8 : Kafka Consumer Adapter

## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_08_kafka_consumer_adapter_code_reference.md`


In [1]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.kafka_consumer_adapter import (
    run_kafka_consumer_to_postgres_once, 
    run_kafka_consumer_to_postgres_loop,
)

from utils.core.env_helpers import (
    env_bool,
    env_float,
    env_int,
    env_optional_int,
    env_str,
)



In [2]:
SCHEMA = env_str(
    "CAPSTONE_SCHEMA",
    "capstone",
    aliases=("CONSUMER_SCHEMA",),
)

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)
RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

OBSERVATION_BATCH_SIZE = env_int(
    "OBSERVATION_BATCH_SIZE",
    500,
    aliases=("SYNTHETIC_OBSERVATION_BATCH_SIZE",),
)

TOPIC = env_str(
    "SYNTHETIC_KAFKA_TOPIC",
    "pump.telemetry.synthetic",
    aliases=("KAFKA_TOPIC",),
)

CONSUMER_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_CONSUMED_MESSAGES_TABLE",
    "synthetic_sensor_messages_consumed_stage",
    aliases=("CONSUMER_TARGET_TABLE",),
)

CONSUMER_GROUP_ID = env_str(
    "SYNTHETIC_CONSUMER_GROUP_ID",
    "synthetic-telemetry-consumer-group",
    aliases=("KAFKA_CONSUMER_GROUP_ID", "CONSUMER_GROUP_ID"),
)

CONSUMER_WORKER_ID = env_str(
    "SYNTHETIC_CONSUMER_WORKER_ID",
    "consumer_worker_001",
    aliases=("CONSUMER_WORKER_ID",),
)

DEFAULT_CONSUMER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

CONSUMER_BATCH_SIZE = env_int(
    "CONSUMER_BATCH_SIZE",
    DEFAULT_CONSUMER_BATCH_SIZE,
)

CONSUMER_POLL_TIMEOUT_SECONDS = env_float(
    "CONSUMER_POLL_TIMEOUT_SECONDS",
    1.0,
)

AUTO_OFFSET_RESET = env_str(
    "CONSUMER_AUTO_OFFSET_RESET",
    "earliest",
    aliases=("SYNTHETIC_AUTO_OFFSET_RESET",),
)

MAX_BATCHES = env_optional_int(
    "CONSUMER_MAX_BATCHES_LIMIT",
    default=100000,
)

IDLE_SLEEP_SECONDS = env_float(
    "CONSUMER_IDLE_SLEEP_SECONDS",
    0.0,
)

STOP_ON_EMPTY_FLAG = env_bool(
    "CONSUMER_STOP_ON_EMPTY",
    True,
)

print("Kafka consumer config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("topic:", TOPIC)
print("target table:", CONSUMER_DESTINATION_TABLE_NAME)
print("consumer group id:", CONSUMER_GROUP_ID)
print("consumer worker id:", CONSUMER_WORKER_ID)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("message batch size:", CONSUMER_BATCH_SIZE)
print("max batches:", MAX_BATCHES)
print("stop on empty:", STOP_ON_EMPTY_FLAG)

Kafka consumer config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
topic: pump.telemetry.synthetic
target table: synthetic_sensor_messages_consumed_stage
consumer group id: synthetic-telemetry-consumer-group
consumer worker id: consumer_worker_test_001
observation batch size: 500
message batch size: 5200
max batches: None
stop on empty: False


---

In [3]:
engine = get_engine_from_env()

---

#### Single Batch

In [ ]:
result = run_kafka_consumer_to_postgres_once(
    engine=engine,
    topic=TOPIC,
    schema=SCHEMA,
    table_name=CONSUMER_DESTINATION_TABLE_NAME,
    max_messages=CONSUMER_BATCH_SIZE,
    poll_timeout_seconds=CONSUMER_POLL_TIMEOUT_SECONDS,
    consumer_group_id=CONSUMER_GROUP_ID,
    consumer_worker_id=CONSUMER_WORKER_ID,
    auto_offset_reset=AUTO_OFFSET_RESET,
)

display(result)

----

#### Loop Batches

In [ ]:
results = run_kafka_consumer_to_postgres_loop(
    engine=engine,
    topic=TOPIC,
    schema=SCHEMA,
    table_name=CONSUMER_DESTINATION_TABLE_NAME,
    max_messages=CONSUMER_BATCH_SIZE,
    poll_timeout_seconds=CONSUMER_POLL_TIMEOUT_SECONDS,
    consumer_group_id=CONSUMER_GROUP_ID,
    consumer_worker_id=CONSUMER_WORKER_ID,
    auto_offset_reset=AUTO_OFFSET_RESET,
    max_batches=MAX_BATCHES,
    idle_sleep_seconds=IDLE_SLEEP_SECONDS,
    stop_on_empty=STOP_ON_EMPTY_FLAG,
)

display(results)

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT kafka_topic || ':' || kafka_partition || ':' || kafka_offset) AS distinct_kafka_messages,
        MIN(consumer_received_at) AS min_consumer_received_at,
        MAX(consumer_received_at) AS max_consumer_received_at
    FROM {SCHEMA}.{CONSUMER_DESTINATION_TABLE_NAME}
    """
)

display(validation_dataframe)

----

In [9]:
stage_8_progress_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS consumed_row_count,
        11700000 - COUNT(*) AS remaining_message_count,
        ROUND((COUNT(*)::numeric / 11700000::numeric) * 100, 2) AS percent_complete,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(*) FILTER (WHERE rebuild_status = 'pending') AS pending_rebuild_count,
        COUNT(*) FILTER (WHERE message_key IS NULL) AS null_message_key_count,
        COUNT(*) FILTER (WHERE sensor_value IS NULL) AS null_sensor_value_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_8_progress_dataframe)

,consumed_row_count,remaining_message_count,percent_complete,distinct_message_key_count,distinct_observation_count,pending_rebuild_count,null_message_key_count,null_sensor_value_count,first_consumed_at,last_consumed_at
0,7675200,4024800,65.6,7675200,147890,7675200,0,0,2026-05-26 22:48:33.489191+00:00,2026-05-26 23:50:31.136768+00:00


---

In [ ]:
consumer_progress_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS consumed_row_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(*) FILTER (WHERE rebuild_status = 'pending') AS pending_rebuild_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(consumer_progress_dataframe)

In [ ]:
consumed_global_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS total_consumed_rows,
        COUNT(DISTINCT dataset_id) AS distinct_dataset_count,
        COUNT(DISTINCT run_id) AS distinct_run_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    """
)

display(consumed_global_dataframe)

In [ ]:
consumed_dataset_run_breakdown_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        consumer_group_id,
        consumer_worker_id,
        COUNT(*) AS row_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    GROUP BY dataset_id, run_id, consumer_group_id, consumer_worker_id
    ORDER BY row_count DESC
    LIMIT 25
    """
)

display(consumed_dataset_run_breakdown_dataframe)

In [ ]:
stage_8_global_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS total_consumed_rows,
        COUNT(DISTINCT dataset_id) AS distinct_dataset_count,
        COUNT(DISTINCT run_id) AS distinct_run_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(*) FILTER (WHERE rebuild_status = 'pending') AS pending_rebuild_count,
        COUNT(*) FILTER (WHERE message_key IS NULL) AS null_message_key_count,
        COUNT(*) FILTER (WHERE sensor_value IS NULL) AS null_sensor_value_count,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    """
)

display(stage_8_global_validation_dataframe)

In [ ]:
stage_8_final_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS consumed_row_count,
        COUNT(DISTINCT message_key) AS distinct_message_key_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT sensor_name) AS distinct_sensor_name_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_index_count,
        COUNT(*) FILTER (WHERE rebuild_status = 'pending') AS pending_rebuild_count,
        COUNT(*) FILTER (WHERE is_duplicate = TRUE) AS duplicate_offset_count,
        COUNT(*) FILTER (WHERE message_key IS NULL) AS null_message_key_count,
        COUNT(*) FILTER (WHERE sensor_value IS NULL) AS null_sensor_value_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(consumer_received_at) AS first_consumed_at,
        MAX(consumer_received_at) AS last_consumed_at,
        (
            COUNT(*) = 11700000
            AND COUNT(DISTINCT message_key) = 11700000
            AND COUNT(DISTINCT observation_index) = 225000
            AND COUNT(DISTINCT sensor_index) = 52
            AND COUNT(*) FILTER (WHERE rebuild_status = 'pending') = 11700000
            AND COUNT(*) FILTER (WHERE message_key IS NULL) = 0
            AND COUNT(*) FILTER (WHERE sensor_value IS NULL) = 0
        ) AS ready_for_stage_9
    FROM "{SCHEMA}"."synthetic_sensor_messages_consumed_stage"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_8_final_validation_dataframe)

In [ ]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

The consumer adapter lands Kafka messages into the consumed-message staging path.
